# 04 — Determinants, Matrix Inverses & Solving Equations
## From Ax = b to the OLS Solution

> **Prerequisites:** Notebooks 01–03  
> **Difficulty:** ⭐⭐⭐☆☆ Intermediate  
> **Running example:** Solving for house price weights (linear regression)

---

## What problem are we solving today?

In the previous notebook, you saw how to do the **forward pass** of a neural network: given inputs X and weights W, compute outputs Z = XW.

But in machine learning, we usually have the **opposite problem**:
- We have inputs X ✓
- We have the desired outputs y ✓
- We need to **find** the weights W that make XW ≈ y

This is **solving a system of equations**: Xw = y, find w.

Three tools help us do this:
1. **Determinant** — tells us *whether* a solution exists
2. **Inverse** — the "undo" button for matrix multiplication
3. **np.linalg.solve** — the practical, efficient way to find the solution

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

---
## 1. The Determinant — Does This Matrix Have a Solution?

### Plain English analogy
Imagine you're using a projection screen. The screen transforms 3D coordinates to 2D coordinates on the wall.

Once you project onto the wall (2D), you've **lost information** — you can't tell what the original 3D position was just from the shadow. This transformation is **not reversible**.

The **determinant** is a number that tells you whether a matrix is reversible:
- **det ≠ 0** → reversible (you can undo the transformation)
- **det = 0** → not reversible (information was lost, like the shadow on the wall)

### For 2D: geometric meaning
For a 2×2 matrix, the determinant = **the area of the parallelogram** formed by the two row vectors.
```
A = [[a, b],
     [c, d]]
det(A) = a×d - b×c
```

- det > 0: transformation preserves orientation
- det < 0: transformation flips orientation (like a mirror)
- det = 0: the parallelogram collapsed to a line (area = 0) — **information is lost!**

In [ ]:
# ─────────────────────────────────────────────────────────
# DETERMINANT: detecting invertibility
# ─────────────────────────────────────────────────────────

print("=== Determinants ===")
print()

# Invertible matrix (det ≠ 0)
A = np.array([[3., 1.],
              [1., 2.]])
det_A = np.linalg.det(A)
print(f"A = \n{A}")
print(f"det(A) = {det_A:.4f}  ← non-zero → A is invertible")
print(f"Manual: 3×2 - 1×1 = {3*2 - 1*1}")

print()
# Singular matrix (det = 0) — NOT invertible
S = np.array([[1., 2.],
              [2., 4.]])   # row 2 = 2 × row 1
det_S = np.linalg.det(S)
print(f"S = \n{S}")
print(f"det(S) = {det_S:.6f}  ← near zero → S is NOT invertible")
print(f"Why: row 2 = 2 × row 1. The two rows are 'the same direction'")
print(f"     → redundant info → cannot be inverted")

print()
# Rotation matrix (det = 1, always invertible — rotations are reversible!)
theta = np.pi / 4  # 45 degrees
R = np.array([[np.cos(theta), -np.sin(theta)],
              [np.sin(theta),  np.cos(theta)]])
print(f"Rotation 45° matrix R = \n{R.round(4)}")
print(f"det(R) = {np.linalg.det(R):.4f}  ← exactly 1 (rotation is always reversible!)")

In [ ]:
# ─────────────────────────────────────────────────────────
# VISUAL: det = area of parallelogram
# ─────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

matrices = [
    (np.array([[2., 1.], [0., 2.]]),  "det = 4 > 1\nExpands area"),
    (np.array([[1., 0.], [0., 1.]]),  "det = 1\nPreserves area (identity)"),
    (np.array([[1., 2.], [0., 0.]]),  "det = 0\nCollapses to a line!"),
]

corners = np.array([[0,0],[1,0],[1,1],[0,1],[0,0]]).T  # unit square

for ax, (M, title) in zip(axes, matrices):
    transformed = M @ corners
    det = np.linalg.det(M)
    
    ax.fill(corners[0], corners[1], alpha=0.3, color='gray', label='original (area=1)')
    ax.plot(corners[0], corners[1], 'gray', lw=1.5, linestyle='--')
    
    ax.fill(transformed[0], transformed[1], alpha=0.4, color='steelblue',
            label=f'transformed (area=|{det:.0f}|)')
    ax.plot(transformed[0], transformed[1], 'steelblue', lw=2)
    
    ax.axhline(0, color='k', lw=0.5); ax.axvline(0, color='k', lw=0.5)
    ax.set_xlim(-0.5, 3.5); ax.set_ylim(-0.5, 3.5)
    ax.set_aspect('equal'); ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)
    ax.set_title(f"{title}\ndet = {det:.1f}", fontsize=10)

plt.suptitle('Determinant = How Much the Matrix Scales Area\ndet=0 means information is lost!', fontsize=12)
plt.tight_layout()
plt.show()

---
## 2. Matrix Inverse — The "Undo" Button

### Plain English
The **inverse** of matrix A, written A⁻¹, undoes what A does:
> A × A⁻¹ = A⁻¹ × A = I  (the identity matrix)

Just like: 5 × (1/5) = 1.

Analogy: If A is "double everything", then A⁻¹ is "halve everything".
If A is "rotate 45°", then A⁻¹ is "rotate -45°".

### When does it exist?
**Only when det(A) ≠ 0.**

### For 2×2 matrices, there's a simple formula:
```
A = [[a, b],    A⁻¹ = 1/det × [[ d, -b],
     [c, d]]                    [-c,  a]]

det = a×d - b×c
```

### Why do we care in ML?
- **Linear regression solution**: w = (XᵀX)⁻¹Xᵀy — this is the formula that finds the best-fit weights!
- **Precision matrix in Gaussian distributions**: Σ⁻¹
- However: in practice, **we rarely compute inverses directly** — there are better methods (see below)

In [ ]:
# ─────────────────────────────────────────────────────────
# MATRIX INVERSE
# ─────────────────────────────────────────────────────────

A = np.array([[3., 1.],
              [1., 2.]])

A_inv = np.linalg.inv(A)
print("=== Matrix Inverse ===")
print(f"A:\n{A}")
print(f"\nA⁻¹:\n{A_inv.round(4)}")
print()
print(f"A @ A⁻¹ (should be identity I):")
print(np.round(A @ A_inv, 10))
print()
print(f"A⁻¹ @ A (should be identity I):")
print(np.round(A_inv @ A, 10))

print()
print("=== Manual formula for 2×2 ===")
a, b, c, d = 3., 1., 1., 2.
det = a*d - b*c
print(f"det = {a}×{d} - {b}×{c} = {det}")
A_inv_manual = (1/det) * np.array([[d, -b], [-c, a]])
print(f"A⁻¹ = (1/{det}) × [[{d}, {-b}], [{-c}, {a}]] =\n{A_inv_manual.round(4)}")
print(f"Matches np.linalg.inv? {np.allclose(A_inv, A_inv_manual)}")

print()
print("=== Singular matrix has no inverse ===")
S = np.array([[1., 2.], [2., 4.]])
print(f"det(S) = {np.linalg.det(S):.6f}  ← near 0")
try:
    S_inv = np.linalg.inv(S)
    print(f"np.linalg.inv still returns something (not always an error):")
    print(S_inv)  # NumPy may not raise, just give garbage
    print("→ But the result is unreliable (det=0 means no true inverse)")
except Exception as e:
    print(f"Error: {e}")

---
## 3. Systems of Linear Equations — Finding the Unknowns

### The core problem in ML
Given a system of equations:
```
2x +  y = 5
 x + 3y = 7
```
We want to find x and y. In matrix form this is: **Ax = b**

```
A = [[2, 1],    x = [x],    b = [5]
     [1, 3]]        [y]         [7]

Find x such that A @ x = b.
```

### Three possible situations
| det(A) | Meaning | Number of solutions |
|---|---|---|
| ≠ 0 | Unique solution exists | **Exactly 1** |
| = 0, b reachable | Infinitely many directions | **Infinite** |
| = 0, b not reachable | No combination of columns reaches b | **None** |

### How to solve: use `np.linalg.solve(A, b)`
**Do NOT compute A⁻¹ and multiply** — it's slower and less numerically stable.
Always prefer `np.linalg.solve` for square systems.

In [ ]:
# ─────────────────────────────────────────────────────────
# SOLVING Ax = b
# ─────────────────────────────────────────────────────────

print("=== Solving a system of equations ===")
print("Problem: Find x, y, z such that:")
print("  2x +  y + 0z = 5")
print("   x + 3y + 2z = 11")
print("  0x +  y + 4z = 9")
print()

A = np.array([[2., 1., 0.],
              [1., 3., 2.],
              [0., 1., 4.]])
b = np.array([5., 11., 9.])

print(f"Matrix A:\n{A}")
print(f"Right-hand side b: {b}")
print(f"det(A) = {np.linalg.det(A):.4f}  ← non-zero → unique solution exists")
print()

# Solve using linalg.solve (preferred method)
solution = np.linalg.solve(A, b)
print(f"Solution: x = {solution}")
print(f"  x = {solution[0]:.3f}")
print(f"  y = {solution[1]:.3f}")
print(f"  z = {solution[2]:.3f}")
print()

# Verify: A @ x should equal b
check = A @ solution
print(f"Verification: A @ x = {check}")
print(f"Original b:         {b}")
print(f"Match: {np.allclose(check, b)}  ← yes!")
print()

# Why NOT to use inverse directly
print("--- Why not use A⁻¹ @ b ---")
solution_via_inv = np.linalg.inv(A) @ b
print(f"Via A⁻¹ @ b: {solution_via_inv}  (same answer, but slower and less stable)")

---
## 4. Real ML Application: Linear Regression

### The problem
We have:
- **X**: dataset matrix (n samples × p features)
- **y**: labels (n prices)
- **w**: weights we want to find (p values)

We want **w** such that **Xw ≈ y** (approximately equal, because real data has noise).

### The Ordinary Least Squares (OLS) solution
The best weights (minimizing squared error) are:
> **w = (XᵀX)⁻¹ Xᵀy**

But we don't compute the inverse explicitly. Instead:
> **solve (XᵀX) w = Xᵀy**

This is *exactly* `np.linalg.solve(X.T @ X, X.T @ y)`.

This formula comes from calculus (setting the gradient of the error to zero), but **all the computation is linear algebra**.

In [ ]:
# ─────────────────────────────────────────────────────────
# LINEAR REGRESSION via solving Ax = b
# ─────────────────────────────────────────────────────────

print("=== Linear Regression: Finding the Best Weights ===")

# Generate synthetic house data
np.random.seed(42)
n = 100
sizes    = np.random.uniform(800, 3000, n)
bedrooms = np.random.randint(1, 6, n).astype(float)

# True relationship: price = 150*size + 25000*bedrooms + 30000 + noise
true_weights = np.array([150.0, 25000.0])
true_bias    = 30000.0
y = true_weights[0]*sizes + true_weights[1]*bedrooms + true_bias + np.random.randn(n)*15000

# Build feature matrix X with bias column (column of 1s for the intercept)
# This trick lets us absorb the bias into the weight vector
X_design = np.column_stack([
    np.ones(n),    # bias column (always 1 for every sample)
    sizes,          # feature 1: size
    bedrooms        # feature 2: bedrooms
])  # shape (100, 3)

print(f"X_design shape: {X_design.shape}  (100 samples, 3: [1, size, bedrooms])")
print(f"y shape:        {y.shape}")
print()

# OLS solution: solve (XᵀX)w = Xᵀy
XtX = X_design.T @ X_design    # (3×100) @ (100×3) = (3×3)
Xty = X_design.T @ y           # (3×100) @ (100,) = (3,)

weights = np.linalg.solve(XtX, Xty)  # solve the 3×3 system

print(f"=== Results ===")
print(f"True  : bias={true_bias:.0f}, w_size={true_weights[0]:.1f}, w_beds={true_weights[1]:.0f}")
print(f"Found : bias={weights[0]:.0f}, w_size={weights[1]:.2f}, w_beds={weights[2]:.0f}")
print()
print(f"Interpretation:")
print(f"  Each extra sqft adds ${weights[1]:.2f} to the price")
print(f"  Each extra bedroom adds ${weights[2]:.0f} to the price")
print(f"  Base price (empty lot): ${weights[0]:.0f}")

# Evaluate
y_pred = X_design @ weights
rmse = np.sqrt(np.mean((y - y_pred)**2))
print(f"\nRMSE: ${rmse:,.0f}  (average prediction error)")

In [ ]:
# ─────────────────────────────────────────────────────────
# VISUALIZE REGRESSION RESULTS
# ─────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Plot 1: predicted vs actual
axes[0].scatter(y/1000, y_pred/1000, alpha=0.5, s=40, color='steelblue')
min_val = min(y.min(), y_pred.min()) / 1000
max_val = max(y.max(), y_pred.max()) / 1000
axes[0].plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Perfect prediction')
axes[0].set_xlabel('Actual Price ($000s)')
axes[0].set_ylabel('Predicted Price ($000s)')
axes[0].set_title('Actual vs Predicted\n(OLS from solving Ax=b)', fontsize=10)
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# Plot 2: size vs price with fit line (at 3 bedrooms)
bedroom_fixed = 3
x_line = np.linspace(sizes.min(), sizes.max(), 100)
y_line = weights[0] + weights[1]*x_line + weights[2]*bedroom_fixed

# Color by bedroom count
scatter = axes[1].scatter(sizes, y/1000, c=bedrooms, cmap='viridis', alpha=0.6, s=40)
axes[1].plot(x_line, y_line/1000, 'r-', lw=2.5,
             label=f'Fit at {bedroom_fixed} bedrooms')
plt.colorbar(scatter, ax=axes[1], label='Bedrooms')
axes[1].set_xlabel('Size (sqft)')
axes[1].set_ylabel('Price ($000s)')
axes[1].set_title('Price vs Size\n(colored by bedrooms)', fontsize=10)
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle('Linear Regression = Solving a System of Linear Equations', fontsize=12)
plt.tight_layout()
plt.show()

---
## 5. What If You Have More Equations Than Unknowns?

### The overdetermined system
In real ML, you have 1000 data points but only 3 weight parameters.
That means 1000 equations and 3 unknowns — **there's no exact solution**.

The best you can do is find weights that **minimize the total error**. This is the **least squares** problem, solved by `np.linalg.lstsq`.

This is what `np.linalg.solve` does automatically when you pass the normal equations (XᵀX)w = Xᵀy — because XᵀX is square (p×p), even though X is tall (n×p).

In [ ]:
# ─────────────────────────────────────────────────────────
# lstsq: when you can't get an exact solution
# ─────────────────────────────────────────────────────────

print("=== np.linalg.lstsq: Least Squares ===")
print("Use this when X is tall (more rows than columns)")
print()

# Directly solve the overdetermined system X_design @ w ≈ y
# (100 equations, 3 unknowns — no exact solution because of noise)
weights_lstsq, residuals, rank, sv = np.linalg.lstsq(X_design, y, rcond=None)

print(f"X_design shape: {X_design.shape}  (overdetermined: 100 equations, 3 unknowns)")
print(f"Weights from lstsq: {weights_lstsq.round(2)}")
print(f"Weights from solve: {weights.round(2)}")
print(f"Agree: {np.allclose(weights_lstsq, weights, rtol=1e-4)}  ← same answer!")
print()
print(f"Matrix rank: {rank}  (number of truly independent equations)")
print(f"Singular values: {sv.round(1)}")
print()
print("When to use which:")
print("  np.linalg.solve  → square A, need exact solution")
print("  np.linalg.lstsq  → any shape A, minimizes ||Ax - b||² (least squares)")

---
## Summary

| Tool | Use it when | NumPy function |
|---|---|---|
| **Determinant** | Check if a matrix is invertible | `np.linalg.det(A)` |
| **Inverse** | Rarely compute directly | `np.linalg.inv(A)` |
| **Solve** | Square system, unique solution | `np.linalg.solve(A, b)` |
| **Least squares** | More data than unknowns (the ML case) | `np.linalg.lstsq(A, b)` |

### The golden rules
1. **det = 0** → no inverse, no unique solution → something is wrong (redundant features?)
2. **Never** compute `np.linalg.inv(A) @ b` — always use `np.linalg.solve(A, b)` instead
3. **Linear regression** = solving (XᵀX)w = Xᵀy — it's all matrix operations!

**Next: Notebook 05 — Eigenvalues & Eigenvectors** (finding the natural directions in data)